In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os
import gc

In [2]:
# Define data loading parameters
DATA_PATH = './IDS2018'
CHUNK_SIZE = 50000  # Process data in chunks to manage memory

In [4]:
# # Optimized function to get all CSV files
# def get_file_paths(data_path):
#     """Get all CSV file paths efficiently"""
#     file_paths = []
    
#     for root, _, filenames in os.walk(data_path):
#         for filename in filenames:
#             if filename.endswith('.csv'):
#                 file_paths.append(os.path.join(root, filename))
#     file_paths.remove('./IDS2018/02-20-2018.csv')
#     print(f"Found {len(file_paths)} CSV files")
#     return sorted(file_paths)

# # Optimized data loading function
# def load_and_clean_data(file_paths):
#     """Load and clean data from multiple CSV files efficiently"""
#     # Labels to exclude (too few samples, sum as 'Anomoly')
#     exclude_labels = ['DoS attacks-Slowloris', 'DoS attacks-GoldenEye', 'DDOS attack-LOIC-UDP', 
#                      'Brute Force -Web', 'Brute Force -XSS', 'SQL Injection']
    
#     # Initialize label encoder
#     label_encoder = LabelEncoder()
#     all_labels = []
    
#     # First pass: collect all unique labels
#     print("First pass: Collecting unique labels...")
#     for file_path in file_paths:
#         print(f"Scanning: {os.path.basename(file_path)}")
#         # Read only the Label column to save memory
#         labels = pd.read_csv(file_path, usecols=['Label'], low_memory=False)
#         all_labels.extend(labels['Label'].unique())
    
#     # Get unique labels and fit encoder
#     unique_labels = list(set(all_labels) - set(exclude_labels))
    
#     if 'Label' in unique_labels:
#         unique_labels.remove('Label')
        
#     excluded_found = any(label in all_labels for label in exclude_labels)
#     if excluded_found:
#         unique_labels.append('Anomoly')
        
#     print(unique_labels)
#     label_encoder.fit(unique_labels)
#     print(f"Found {len(unique_labels)} unique attack types")
    
#     # Second pass: load and process data
#     processed_chunks = []
    
#     for file_path in file_paths:
#         print(f"Processing: {os.path.basename(file_path)}")
        
#         # Read in chunks to manage memory
#         chunk_reader = pd.read_csv(file_path, chunksize=CHUNK_SIZE, low_memory=False)
        
#         for chunk in chunk_reader:
#             # Remove timestamp and excluded labels
#             chunk = chunk.drop(columns=['Timestamp'], errors='ignore')
            
#             if len(chunk) > 0:
                
#                 chunk = chunk[chunk['Label'] != 'Label']
#                 chunk['Label'] = chunk['Label'].apply(lambda x: 'Anomoly' if x in exclude_labels else x)
                
#                 # Encode labels
#                 chunk['Label'] = label_encoder.transform(chunk['Label'])
                
#                 # Convert to numeric and clean
#                 chunk = chunk.apply(pd.to_numeric, errors='coerce')
#                 chunk = chunk.replace([np.inf, -np.inf], np.nan)
#                 chunk = chunk.dropna()
                
#                 processed_chunks.append(chunk)

#                 if len(processed_chunks) % 10 == 0:  # Every 10 chunks
#                     gc.collect()
        
#         # Clear memory
#         gc.collect()
    
#     # Combine all chunks
#     print("Combining all processed chunks...")
#     combined_df = pd.concat(processed_chunks, ignore_index=True)
#     del processed_chunks
#     gc.collect()
    
#     return combined_df, label_encoder

# # Get file paths
# file_paths = get_file_paths(DATA_PATH)

# # Load and clean data
# print("Loading and cleaning data...")
# df, label_encoder_fitted = load_and_clean_data(file_paths)

# _=gc.collect()

import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ===== SETTINGS =====
DATA_PATH = "./IDS2018"      # Folder containing raw CSVs
CHUNK_SIZE = 50000           # Read in chunks
OUTPUT_FILE = os.path.join(DATA_PATH, "IDS2018_combined.csv")
MAP_FILE = os.path.join(DATA_PATH, "label_mapping.csv")

# ===== 1. Find CSV files =====
def get_file_paths(data_path):
    file_paths = []
    for root, _, files in os.walk(data_path):
        for file in files:
            if file.endswith(".csv"):
                file_paths.append(os.path.join(root, file))
    # Remove 02-20-2018.csv if present
    skip_file = os.path.join(data_path, "02-20-2018.csv")
    if skip_file in file_paths:
        file_paths.remove(skip_file)
    return sorted(file_paths)

file_paths = get_file_paths(DATA_PATH)
print(f"Found {len(file_paths)} CSV files")

# ===== 2. First pass — collect unique labels =====
print("Loading and cleaning data...")
print("First pass: Collecting unique labels...")
labels = []
for fp in file_paths:
    print(f"Scanning: {os.path.basename(fp)}")
    for chunk in pd.read_csv(fp, chunksize=CHUNK_SIZE, low_memory=False):
        if "Label" in chunk.columns:
            labels.extend(chunk["Label"].dropna().astype(str).unique().tolist())

labels = sorted(set(labels))
print(labels)
print(f"Found {len(labels)} unique attack types")

# ===== 3. Fit LabelEncoder =====
le = LabelEncoder()
le.fit(labels)

# Save label mapping
pd.DataFrame({
    "encoded_value": range(len(le.classes_)),
    "label_name": le.classes_
}).to_csv(MAP_FILE, index=False)

# ===== 4. Second pass — process and write incrementally =====
print("Processing files...")
first_write = True  # Flag for writing header once

for fp in file_paths:
    print(f"Processing: {os.path.basename(fp)}")
    for chunk in pd.read_csv(fp, chunksize=CHUNK_SIZE, low_memory=False):
        # Clean column names
        chunk.columns = [c.strip() for c in chunk.columns]

        # Convert non-label columns to numeric
        for col in chunk.columns:
            if col != "Label":
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

        # Encode labels
        chunk["Label"] = chunk["Label"].astype(str)
        unknown_mask = ~chunk["Label"].isin(le.classes_)
        if unknown_mask.any():
            chunk.loc[unknown_mask, "Label"] = le.classes_[0]
        chunk["Label"] = le.transform(chunk["Label"])

        # Downcast numerics to save space
        num_cols = chunk.select_dtypes(include=["float64", "int64"]).columns
        chunk[num_cols] = chunk[num_cols].apply(pd.to_numeric, downcast="float")
        chunk[num_cols] = chunk[num_cols].apply(pd.to_numeric, downcast="integer")

        # Append to output CSV
        chunk.to_csv(OUTPUT_FILE, mode="a", index=False, header=first_write)
        first_write = False  # Only write header for first chunk

print(f"✅ All processed data saved to {OUTPUT_FILE}")


Found 9 CSV files
Loading and cleaning data...
First pass: Collecting unique labels...
Scanning: 02-14-2018.csv
Scanning: 02-15-2018.csv
Scanning: 02-16-2018.csv
Scanning: 02-21-2018.csv
Scanning: 02-22-2018.csv
Scanning: 02-23-2018.csv
Scanning: 02-28-2018.csv
Scanning: 03-01-2018.csv
Scanning: 03-02-2018.csv
['Benign', 'Bot', 'Brute Force -Web', 'Brute Force -XSS', 'DDOS attack-HOIC', 'DDOS attack-LOIC-UDP', 'DoS attacks-GoldenEye', 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest', 'DoS attacks-Slowloris', 'FTP-BruteForce', 'Infilteration', 'Label', 'SQL Injection', 'SSH-Bruteforce']
Found 15 unique attack types
Processing files...
Processing: 02-14-2018.csv


/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_resul

Processing: 02-15-2018.csv


/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_resul

Processing: 02-16-2018.csv
Processing: 02-21-2018.csv
Processing: 02-22-2018.csv


/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_resul

Processing: 02-23-2018.csv


/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)
/home/ritvik/Code/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:377: RuntimeWarning: invalid value encountered in cast
  new_resul

KeyboardInterrupt: 

In [ ]:
print("Label encoder mapping:")
for i, label in enumerate(label_encoder_fitted.classes_):
    print(f"{i}: {label}")

In [ ]:
print("Saving as CSV...")
mapping_df = pd.DataFrame({
    'encoded_value': range(len(label_encoder_fitted.classes_)),
    'label_name': label_encoder_fitted.classes_
})
mapping_df.to_csv('/IDS2018.csv', index=False)
print("Mapping saved as '/IDS2018.csv'")

In [ ]:
# Display data information
print(f"Total dataset shape: {df.shape}")
print(f"Number of classes: {df['Label'].nunique()}")
print("Label distribution:")
print(df['Label'].value_counts().sort_index())
print(f"Memory usage: {df.memory_usage().sum() / 1024**2:.2f} MB")